<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

```
Input: Soil layers (θ, ψ, K, b, Δz), root parameters, LAI
                    │
                    ▼
    ┌────────────────────────────────────────┐
    │  For each soil layer j = 1 to 11:      │
    │                                        │
    │  1. K_soil = Ksat · s^(2b+3)           │
    │     (soil hydraulic conductivity)      │
    │                                        │
    │  2. RLD = root biomass / (ρ · π·r²·Δz) │
    │     (root length density)              │
    │                                        │
    │  3. r_dist = 1/√(π·RLD)                │
    │     (half-distance between roots)      │
    │                                        │
    │  4. R_soil = ln(r_dist/r_root)         │
    │            / (2π·RLD·Δz·K)             │
    │     (soil-to-root radial resistance)   │
    │                                        │
    │  5. R_root = ρ_resist / (ρ_bio · Δz)   │
    │     (root tissue resistance)           │
    │                                        │
    │  6. R_layer = R_soil + R_root  (series)│
    │                                        │
    │  7. Sum conductances: Σ 1/R_layer      │
    │     (parallel across layers)           │
    │                                        │
    │  8. E_max,j = (ψ_soil,j − ψ_min)/R_j   │
    │     (max transpiration from layer)     │
    └────────────────────────────────────────┘
                    │
                    ▼
    ┌────────────────────────────────────────┐
    │  R_total = LAI / Σ(1/R_layer)          │
    │  (convert ground → leaf area)          │
    └────────────────────────────────────────┘
                    │
                    ▼
    ┌────────────────────────────────────────┐
    │  ψ_soil = Σ(ψ_j · E_max,j) / Σ E_max   │
    │  (transpiration-weighted potential)    │
    └────────────────────────────────────────┘
                    │
                    ▼
    ┌────────────────────────────────────────┐
    │  et_loss_j = E_max,j / Σ E_max         │
    │  (fractional uptake per layer)         │
    └────────────────────────────────────────┘
                    │
                    ▼
          Return rsoil, psi_soil, et_loss
```

In [0]:
#| echo: false
#| output: asis
show_doc(soil_resistance)

---

[source](https://github.com/ecamo19/plant_hydraulics/blob/main/plant_hydraulics/soil_resistance.py#L11){target="_blank" style="float:right; font-size:smaller"}

### soil_resistance

```python

def soil_resistance(
    physcon:PhysCon, # Physical constants:
- grav : float
    Gravitational acceleration (m/s2).
- denh2o : float
    Density of liquid water (kg/m3).
- mmh2o : float
    Molecular mass of water (kg/mol).
    leaf:Leaf, # Leaf parameters:
- minl_wp : float
    Minimum leaf water potential (MPa).
    rootvar:RootVar, # Fine root parameters:
- biomass : float
    Fine root biomass (g biomass/m2).
- radius : float
    Fine root radius (m).
- density : float
    Fine root density (g biomass/m3 root).
- resist : float
    Hydraulic resistivity of root tissue
    (MPa · s · g/mmol H2O).
    soil:Soil, # Soil profile variables:
- nlevsoi : int
    Number of soil layers.
- h2osoi_vol : list of float
    Volumetric water content for each layer (m3/m3).
- psi : list of float
    Matric potential for each layer (mm).
- watsat : list of float
    Volumetric water content at saturation (porosity)
    for each layer (m3/m3).
- hksat : list of float
    Hydraulic conductivity at saturation for each layer
    (mm H2O/s).
- bsw : list of float
    Clapp and Hornberger "b" parameter for each layer (-).
- rootfr : list of float
    Fraction of roots in each layer (-).
- dz : list of float
    Thickness of each layer (m).
    flux:Flux, # Flux variables:
- lai : float
    Canopy leaf area index (m2/m2).
)->Flux: # Updated flux object with the following attributes:
- rsoil : float
    Soil hydraulic resistance (MPa · s · m2/mmol H2O).
- psi_soil : float
    Weighted soil water potential (MPa).
- et_loss : list of float
    Fraction of total transpiration from each soil layer (-).


```

*Calculate soil hydraulic resistance, weighted soil water potential,*
and water uptake from each soil layer.

For each soil layer, computes the belowground resistance as the sum
of a soil-to-root radial resistance (based on soil hydraulic
conductivity and root geometry) and a root-to-stem axial resistance
(based on root tissue resistivity). Layers are treated as parallel
pathways, so their conductances (1/resistance) are summed to obtain
the total belowground conductance. The weighted soil water potential
is computed by weighting each layer's matric potential by its
maximum transpiration capacity.

The soil-to-root radial resistance for each layer is:

    R_soil = ln(r_dist / r_root) / (2π · RLD · Δz · K_soil)

where r_dist is the mean half-distance between roots, RLD is root
length density, Δz is layer thickness, and K_soil is the
unsaturated hydraulic conductivity. The root-to-stem axial
resistance is:

    R_root = ρ_root / (ρ_biomass · Δz)

where ρ_root is the root tissue hydraulic resistivity and ρ_biomass
is the root biomass density. The total belowground resistance per
unit ground area is converted to per unit leaf area by multiplying
by LAI.

The weighted soil water potential uses the maximum transpiration
rate from each layer (ψ_soil,j - ψ_min) / R_j as weights, so that
wetter, better-connected layers contribute more to the effective
soil water supply.

__Parameters:__

- PhysCon : Physical constants:
    - grav : Gravitational acceleration (m/s2).
    - denh2o : Density of liquid water (kg/m3).
    - mmh2o : Molecular mass of water (kg/mol).

- Leaf : Leaf parameters:
    - minl_wp : Minimum leaf water potential (MPa).

- RootVar : Fine root parameters:
    - biomass : Fine root biomass (g biomass/m2).
    - radius : Fine root radius (m).
    - density : Fine root density (g biomass/m3 root).
    - resist : Hydraulic resistivity of root tissue (MPa · s · g/mmol H2O)

- Soil : Soil profile variables:
    - nlevsoi : Number of soil layers.
    - h2osoi_vol : Volumetric water content for each layer (m3/m3).
    - psi : Matric potential for each layer (mm).
    - watsat : Volumetric water content at saturation (porosity)for each
               layer (m3/m3).
    - hksat : Hydraulic conductivity at saturation for each layer (mm H2O/s).
    - bsw : Clapp and Hornberger "b" parameter for each layer (-).
    - rootfr : Fraction of roots in each layer (-).
    - dz : Thickness of each layer (m).

- Flux : Flux variables:
    - lai : Canopy leaf area index (m2/m2).

__Returns:__

- Flux : Updated flux object with the following attributes:
    - rsoil : Soil hydraulic resistance (MPa · s · m2/mmol H2O).
    - psi_soil : Weighted soil water potential (MPa).
    - et_loss : Fraction of total transpiration from each soil layer (-).

#### Example soil_resistance()